# Event Analysis Queries
* Ryan Kazmerik
* April 29, 2025

Proposed queries for basic event analysis, such as the number of events of a user, user event flow in a time window and additional queries that could be interesting to analyze the user behaviour.

In [1]:
import awswrangler as wr
import pandas as pd

In [2]:
DATABASE = "events_db"
S3_BUCKET = "s3://athena-query-results-806a5225/results/"

## Basic Queries
### Events by User
Let's see who are most active users are by querying to see the number of events by user

In [5]:
wr.athena.read_sql_query(
    database= DATABASE,
    s3_output= S3_BUCKET,
    sql= """
        SELECT user_id, COUNT(*) AS total_events
        FROM events_db.events
        GROUP BY user_id
        ORDER BY total_events DESC;
    """
).head()

,user_id,total_events
0,627276,10
1,402028,9
2,939733,8
3,271986,7
4,548966,6


### User Event Flow in a Time Window
Let's dig into the top user and see what activities they've been up to in the past 60 days

In [7]:
wr.athena.read_sql_query(
    database= DATABASE,
    s3_output= S3_BUCKET,
    sql= """
        WITH most_active_user AS (
            SELECT user_id
            FROM events_db.events
            GROUP BY user_id
            ORDER BY COUNT(*) DESC
            LIMIT 1
        )
        SELECT e.page, e.event_type, COUNT(*) AS event_count
        FROM events_db.events e
        JOIN most_active_user m ON e.user_id = m.user_id
        WHERE e.event_timestamp >= current_date - interval '60' day
        GROUP BY e.page, e.event_type
        ORDER BY event_count DESC;
    """
).head()

,page,event_type,event_count
0,tags/posts/tag,view,1
1,wp-content/wp-content/blog,login,1
2,explore/search,purchase,1
3,category/main,click,1


## Additional Queries

### What Content is Driving Purchases?
* Objective: New Customer Acquisition
* Target Audience: Revenue Team 

One thing we're curious about is what's driving traffic to our purchase page? One way to investigate this may be to see what pages users were viewing before they decided to visit the purchase page to help understand what content is making users want to purchase.


In [56]:
wr.athena.read_sql_query(
    database= DATABASE,
    s3_output= S3_BUCKET,
    sql= """
        WITH user_journeys AS (
            SELECT user_id, event_type, event_timestamp, page,
            LAG(page) OVER (PARTITION BY user_id ORDER BY event_timestamp) AS previous_page
            FROM events_db.events
        ),
        product_visits AS (
            SELECT previous_page, COUNT(*) AS product_purchases
            FROM user_journeys
            WHERE page = 'products'
            AND previous_page != 'products'
            AND event_type = 'purchase'
            GROUP BY previous_page
        )
        SELECT previous_page, product_purchases
        FROM product_visits
        ORDER BY product_purchases DESC;
    """
).head(10)

,previous_page,product_purchases
0,settings,9
1,account,8
2,articles,8
3,promotions,7
4,pricing,5
5,login,5
6,sizzle,4


### Which Users are Losing Interest?

* Objective : Customer Retention
* Target Audience : Customer Success Team

It may be helpful to identify users who used to use our app a lot, but their activity is dropping off. We could reach out to them to prevent them from churning from our service all together.

In [39]:
wr.athena.read_sql_query(
    database= DATABASE,
    s3_output= S3_BUCKET,
    sql= """
        WITH past_active_users AS (
            SELECT user_id, COUNT(*) as events_in_past_6m
            FROM events_db.events
            WHERE event_timestamp BETWEEN date_add('day', -180, current_date) AND date_add('day', -30, current_date)
            GROUP BY user_id
        ),
        recent_inactive_users AS (
            SELECT user_id, COUNT(*) as events_this_month
            FROM events_db.events
            WHERE event_timestamp >= date_add('day', -30, current_date)
            GROUP BY user_id
        )
        SELECT p.user_id, events_in_past_6m, events_this_month
        FROM past_active_users p
        LEFT JOIN recent_inactive_users r ON p.user_id = r.user_id
        WHERE r.user_id IS NULL
        ORDER BY events_in_past_6m DESC;
    """
).fillna(0).head(10)

,user_id,events_in_past_6m,events_this_month
0,375970,8,0
1,218912,8,0
2,724485,7,0
3,199516,7,0
4,546557,7,0
5,563499,7,0
6,106717,7,0
7,514166,6,0
8,638506,6,0
9,273318,6,0


### What Platform Drives the most Engagement?

* Objective : Channel Prioritization
* Target Audience : Product Team

We can look at which platform helps drive the most engagement with it's users. This might help inform the product team if one of our channels needs extra attention, or additional investment.

In [43]:
wr.athena.read_sql_query(
    database= DATABASE,
    s3_output= S3_BUCKET,
    sql= """
        WITH recent_events AS (
            SELECT device, date_trunc('month', event_timestamp) AS event_month
            FROM events_db.events
            WHERE event_timestamp >= current_date - interval '6' month
        ),
        monthly_event_counts AS (
            SELECT device, event_month, COUNT(*) AS monthly_events
            FROM recent_events
            GROUP BY device, event_month
        )
        SELECT device, ROUND(AVG(monthly_events), 2) AS avg_events_per_month
        FROM monthly_event_counts
        GROUP BY device
        ORDER BY avg_events_per_month DESC;
    """
).head(10)

,device,avg_events_per_month
0,Mobile,49.71
1,Desktop,47.86
2,Tablet,43.29


### Who's Having Trouble Logging In?

* Objective : Reduce Friction
* Target Audience : Support Team

Let's see if any users are consistently having trouble logging in, maybe we can proactively reach out to them and help them access our app.

In [54]:
wr.athena.read_sql_query(
    database= DATABASE,
    s3_output= S3_BUCKET,
    sql= """
        WITH failed_logins AS (
            SELECT user_id, device, browser, event_timestamp
            FROM events_db.events
            WHERE page = 'login' AND event_type = 'error'
        )
        SELECT user_id, COUNT(*) AS failed_login_attempts
        FROM failed_logins
        GROUP BY user_id
        ORDER BY failed_login_attempts DESC;
    """
).head(10)

,user_id,failed_login_attempts
0,417224,2
1,171236,1
2,451832,1
3,605260,1
4,539010,1
5,568792,1
6,685538,1
7,115806,1
8,433000,1
9,136901,1


### These queries are simple but actionable and help us determine our next best action for improving our app.